# DetectiveQA Baseline 3 — Summary-level Hybrid Routing → Paragraph Rerank

## 1. Install dependencies

In [1]:
!pip -q install -U sentence-transformers faiss-cpu rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 87.8 MB/s eta 0:00:00


## 2. Clone repo and configure paths

In [2]:
from pathlib import Path
import os
import json
import math
import re
import subprocess
from collections import defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

REPO_URL = "https://github.com/TranTheHung2312332/canon-layer.git"
BRANCH = "main"
REPO_DIR = Path("/content/repo")

if "<YOUR_USERNAME>" in REPO_URL:
    raise ValueError("Please set REPO_URL to your lore-router GitHub repo URL.")

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

RANKING_DIR = REPO_DIR / "data" / "detectiveqa-ranking-en"
SUMMARY_DIR = REPO_DIR / "data" / "detectiveqa-summary-en"
OUT_DIR = REPO_DIR / "artifacts" / "retrieval_baselines_v3_summary_hier"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repo:", REPO_DIR)
print("Ranking data:", RANKING_DIR)
print("Summary data:", SUMMARY_DIR)
print("Output:", OUT_DIR)

Cloning into '/content/repo'...
remote: Enumerating objects: 1133, done.
remote: Counting objects: 100% (1133/1133), done.
remote: Compressing objects: 100% (1122/1122), done.
remote: Total 1133 (delta 8), reused 1131 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (1133/1133), 16.33 MiB | 13.23 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Updating files: 100% (1114/1114), done.
Repo: /content/repo
Ranking data: /content/repo/data/detectiveqa-ranking-en
Summary data: /content/repo/data/detectiveqa-summary-en
Output: /content/repo/artifacts/retrieval_baselines_v3_summary_hier


## 3. Experiment config

In [13]:
MAX_QUERIES = 0          # 0 = all queries; use small number for smoke test
CHUNK_TOP_K = 5          # summary-level metric uses k=5
PARAGRAPH_TOP_N = 100     # paragraph candidates before reranking
FINAL_TOP_K = 10         # final paragraph output

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L6-v2"

CHUNK_DENSE_DEPTH = 100
CHUNK_BM25_DEPTH = 100
CHUNK_DENSE_WEIGHT = 1.0
CHUNK_BM25_WEIGHT = 0.5
RRF_K = 60

PARA_DENSE_ALPHA = 0.85  # hybrid paragraph score = alpha*dense + (1-alpha)*bm25
EMBED_BATCH_SIZE = 128
RERANK_BATCH_SIZE = 64

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 4. Load data

In [4]:
def load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_qrels(path: Path):
    qrels = defaultdict(set)
    with path.open("r", encoding="utf-8") as f:
        header = f.readline()
        for line in f:
            parts = line.rstrip("").split("	")
            if len(parts) >= 3 and float(parts[2]) > 0:
                qrels[parts[0]].add(parts[1])
    return dict(qrels)

corpus_rows = load_jsonl(RANKING_DIR / "corpus.jsonl")
query_rows = load_jsonl(RANKING_DIR / "queries.jsonl")
metadata_rows = load_jsonl(RANKING_DIR / "query_metadata.jsonl")
chunk_summary_rows = load_jsonl(SUMMARY_DIR / "chunk_summaries.jsonl")
paragraph_to_chunk_rows = load_jsonl(SUMMARY_DIR / "paragraph_to_chunk.jsonl")

corpus = {row["_id"]: row for row in corpus_rows}
queries = {row["_id"]: row["text"] for row in query_rows}
query_meta = {row["query_id"]: row for row in metadata_rows}
paragraph_qrels = load_qrels(RANKING_DIR / "qrels" / "test.tsv")

chunk_qrels_path = SUMMARY_DIR / "qrels" / "chunks.tsv"
if chunk_qrels_path.exists():
    chunk_qrels = load_qrels(chunk_qrels_path)
else:
    paragraph_to_chunk = {row["paragraph_doc_id"]: row["chunk_id"] for row in paragraph_to_chunk_rows}
    chunk_qrels = {}
    for qid, gold_docs in paragraph_qrels.items():
        chunk_qrels[qid] = {paragraph_to_chunk[d] for d in gold_docs if d in paragraph_to_chunk}

chunk_rows = {row["_id"]: row for row in chunk_summary_rows}
paragraph_to_chunk = {row["paragraph_doc_id"]: row["chunk_id"] for row in paragraph_to_chunk_rows}

print("Paragraphs:", len(corpus))
print("Queries:", len(queries))
print("Chunk summaries:", len(chunk_rows))
print("Paragraph qrel queries:", len(paragraph_qrels))
print("Chunk qrel queries:", len(chunk_qrels))

Paragraphs: 65641
Queries: 149
Chunk summaries: 986
Paragraph qrel queries: 149
Chunk qrel queries: 149


## 5. Build in-memory indexes from metadata

In [5]:
TOKEN_RE = re.compile(r"[A-Za-z0-9]+")

def tokenize(text: str):
    return TOKEN_RE.findall(text.lower())

chunks_by_novel = defaultdict(list)
for row in chunk_summary_rows:
    chunks_by_novel[int(row["novel_id"])].append(row)
for novel_id in chunks_by_novel:
    chunks_by_novel[novel_id].sort(key=lambda r: int(r.get("chunk_index", 0)))

paragraphs_by_novel = defaultdict(list)
for doc_id, row in corpus.items():
    # doc IDs follow dqa-en-{novel_id}-p{paragraph_id}
    novel_id = int(doc_id.split("-")[2])
    paragraphs_by_novel[novel_id].append(row)
for novel_id in paragraphs_by_novel:
    paragraphs_by_novel[novel_id].sort(key=lambda r: int(r["_id"].split("-p")[-1]))

paragraphs_by_chunk = defaultdict(list)
for row in paragraph_to_chunk_rows:
    paragraphs_by_chunk[row["chunk_id"]].append(row["paragraph_doc_id"])

for chunk_id in paragraphs_by_chunk:
    paragraphs_by_chunk[chunk_id].sort(key=lambda d: int(d.split("-p")[-1]))

query_ids = [qid for qid in queries if qid in paragraph_qrels and qid in query_meta]
if MAX_QUERIES:
    query_ids = query_ids[:MAX_QUERIES]

print("Evaluation queries:", len(query_ids))
print("Novels with chunks:", len(chunks_by_novel))

Evaluation queries: 149
Novels with chunks: 23


## 6. Load embedding and reranker models

In [ ]:
import torch
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
reranker = CrossEncoder(RERANKER_MODEL, device=DEVICE)

## 7. Encode summaries and paragraphs

In [ ]:
def encode_texts(texts, batch_size=128, normalize=True):
    return embedder.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=normalize,
        show_progress_bar=True,
    ).astype("float32")

chunk_index_by_novel = {}
paragraph_index_by_novel = {}

for novel_id in tqdm(sorted(chunks_by_novel), desc="Build chunk indexes"):
    rows = chunks_by_novel[novel_id]
    texts = [r.get("summary", "") for r in rows]
    embeddings = encode_texts(texts, batch_size=EMBED_BATCH_SIZE)
    bm25 = BM25Okapi([tokenize(t) for t in texts])
    faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
    faiss_index.add(embeddings)
    chunk_index_by_novel[novel_id] = {
        "rows": rows,
        "texts": texts,
        "embeddings": embeddings,
        "bm25": bm25,
        "faiss": faiss_index,
    }

for novel_id in tqdm(sorted(paragraphs_by_novel), desc="Build paragraph indexes"):
    rows = paragraphs_by_novel[novel_id]
    texts = [r.get("text", "") for r in rows]
    embeddings = encode_texts(texts, batch_size=EMBED_BATCH_SIZE)
    bm25 = BM25Okapi([tokenize(t) for t in texts])
    doc_id_to_pos = {r["_id"]: i for i, r in enumerate(rows)}
    paragraph_index_by_novel[novel_id] = {
        "rows": rows,
        "texts": texts,
        "embeddings": embeddings,
        "bm25": bm25,
        "doc_id_to_pos": doc_id_to_pos,
    }

## 8. Metric functions

In [7]:
def precision_at_k(ranked_ids, gold_ids, k):
    top = ranked_ids[:k]
    if not top:
        return 0.0
    return sum(1 for x in top if x in gold_ids) / k


def recall_at_k(ranked_ids, gold_ids, k):
    if not gold_ids:
        return 0.0
    top = ranked_ids[:k]
    return sum(1 for x in top if x in gold_ids) / len(gold_ids)


def reciprocal_rank(ranked_ids, gold_ids):
    for i, x in enumerate(ranked_ids, start=1):
        if x in gold_ids:
            return 1.0 / i
    return 0.0


def ndcg_at_k(ranked_ids, gold_ids, k):
    dcg = 0.0
    for i, x in enumerate(ranked_ids[:k], start=1):
        if x in gold_ids:
            dcg += 1.0 / math.log2(i + 1)
    ideal_hits = min(len(gold_ids), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def minmax(values):
    values = np.asarray(values, dtype="float32")
    if len(values) == 0:
        return values
    lo, hi = float(values.min()), float(values.max())
    if hi - lo < 1e-9:
        return np.zeros_like(values)
    return (values - lo) / (hi - lo)

## 9. Retrieval functions

In [8]:
def rrf_rank(dense_order, bm25_order, bm25_scores=None, dense_weight=1.0, bm25_weight=0.5, rrf_k=60):
    scores = defaultdict(float)
    for rank, idx in enumerate(dense_order, start=1):
        scores[int(idx)] += dense_weight / (rrf_k + rank)
    for rank, idx in enumerate(bm25_order, start=1):
        if bm25_scores is not None and bm25_scores[int(idx)] <= 0:
            continue
        scores[int(idx)] += bm25_weight / (rrf_k + rank)
    return [idx for idx, _ in sorted(scores.items(), key=lambda kv: kv[1], reverse=True)]


def rank_chunks(query_text, novel_id, top_k=5):
    index = chunk_index_by_novel[novel_id]
    q_emb = encode_texts([QUERY_PREFIX + query_text], batch_size=1)[0]

    dense_scores = index["embeddings"] @ q_emb
    dense_order = np.argsort(-dense_scores)[:CHUNK_DENSE_DEPTH]

    bm25_scores = np.asarray(index["bm25"].get_scores(tokenize(query_text)), dtype="float32")
    bm25_order = np.argsort(-bm25_scores)[:CHUNK_BM25_DEPTH]

    fused_order = rrf_rank(
        dense_order=dense_order,
        bm25_order=bm25_order,
        bm25_scores=bm25_scores,
        dense_weight=CHUNK_DENSE_WEIGHT,
        bm25_weight=CHUNK_BM25_WEIGHT,
        rrf_k=RRF_K,
    )
    return [index["rows"][i]["_id"] for i in fused_order[:top_k]]


def score_paragraphs(query_text, novel_id, candidate_doc_ids, top_n=30):
    index = paragraph_index_by_novel[novel_id]
    candidate_doc_ids = [d for d in candidate_doc_ids if d in index["doc_id_to_pos"]]
    if not candidate_doc_ids:
        return []

    q_emb = encode_texts([QUERY_PREFIX + query_text], batch_size=1)[0]
    positions = np.array([index["doc_id_to_pos"][d] for d in candidate_doc_ids], dtype=np.int64)

    dense_scores = index["embeddings"][positions] @ q_emb
    bm25_all = np.asarray(index["bm25"].get_scores(tokenize(query_text)), dtype="float32")
    bm25_scores = bm25_all[positions]

    dense_norm = minmax(dense_scores)
    bm25_norm = minmax(bm25_scores)
    final_scores = PARA_DENSE_ALPHA * dense_norm + (1.0 - PARA_DENSE_ALPHA) * bm25_norm

    order = np.argsort(-final_scores)[:top_n]
    return [candidate_doc_ids[i] for i in order]


def rerank_paragraphs(query_text, doc_ids):
    if not doc_ids:
        return []
    pairs = [(query_text, corpus[d]["text"]) for d in doc_ids]
    scores = reranker.predict(pairs, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False)
    ranked = [d for d, _ in sorted(zip(doc_ids, scores), key=lambda x: float(x[1]), reverse=True)]
    return ranked

## 10. Run baseline 3

In [14]:
paragraph_metric_rows = []
chunk_metric_rows = []
run_paragraph_rows = []
run_chunk_rows = []

for qid in tqdm(query_ids, desc="Evaluate"):
    query_text = queries[qid]
    novel_id = int(query_meta[qid]["novel_id"])

    selected_chunks = rank_chunks(query_text, novel_id, top_k=CHUNK_TOP_K)
    gold_chunks = chunk_qrels.get(qid, set())
    gold_paragraphs = paragraph_qrels.get(qid, set())

    chunk_metric_rows.append({
        "query_id": qid,
        "novel_id": novel_id,
        "chunk_precision_at_5": precision_at_k(selected_chunks, gold_chunks, CHUNK_TOP_K),
        "chunk_recall_at_5": recall_at_k(selected_chunks, gold_chunks, CHUNK_TOP_K),
        "chunk_mrr": reciprocal_rank(selected_chunks, gold_chunks),
        "chunk_ndcg_at_5": ndcg_at_k(selected_chunks, gold_chunks, CHUNK_TOP_K),
        "gold_chunk_count": len(gold_chunks),
    })

    candidate_doc_ids = []
    seen = set()
    for chunk_id in selected_chunks:
        for doc_id in paragraphs_by_chunk.get(chunk_id, []):
            if doc_id not in seen:
                candidate_doc_ids.append(doc_id)
                seen.add(doc_id)

    top30 = score_paragraphs(query_text, novel_id, candidate_doc_ids, top_n=PARAGRAPH_TOP_N)
    final_ranking = rerank_paragraphs(query_text, top30)[:FINAL_TOP_K]

    paragraph_metric_rows.append({
        "query_id": qid,
        "novel_id": novel_id,
        "precision_at_10": precision_at_k(final_ranking, gold_paragraphs, FINAL_TOP_K),
        "recall_at_10": recall_at_k(final_ranking, gold_paragraphs, FINAL_TOP_K),
        "mrr": reciprocal_rank(final_ranking, gold_paragraphs),
        "ndcg_at_10": ndcg_at_k(final_ranking, gold_paragraphs, FINAL_TOP_K),
        "gold_paragraph_count": len(gold_paragraphs),
        "selected_chunk_count": len(selected_chunks),
        "candidate_paragraph_count": len(candidate_doc_ids),
    })

    run_chunk_rows.append({
        "query_id": qid,
        "novel_id": novel_id,
        "query": query_text,
        "ranked_chunk_ids": selected_chunks,
        "gold_chunk_ids": sorted(gold_chunks),
    })

    run_paragraph_rows.append({
        "query_id": qid,
        "novel_id": novel_id,
        "query": query_text,
        "selected_chunk_ids": selected_chunks,
        "candidate_paragraph_count": len(candidate_doc_ids),
        "top30_before_rerank": top30,
        "top10_after_rerank": final_ranking,
        "gold_doc_ids": sorted(gold_paragraphs),
    })

paragraph_metrics = pd.DataFrame(paragraph_metric_rows)
chunk_metrics = pd.DataFrame(chunk_metric_rows)

paragraph_summary = paragraph_metrics[["precision_at_10", "recall_at_10", "mrr", "ndcg_at_10", "candidate_paragraph_count"]].mean().to_dict()
chunk_summary = chunk_metrics[["chunk_precision_at_5", "chunk_recall_at_5", "chunk_mrr", "chunk_ndcg_at_5"]].mean().to_dict()

summary_rows = [
    {
        "system": "summary_hybrid_chunks5_para30_rerank10",
        "level": "paragraph",
        **paragraph_summary,
    },
    {
        "system": "summary_hybrid_chunks5",
        "level": "chunk_summary",
        **chunk_summary,
    },
]
summary_df = pd.DataFrame(summary_rows)
summary_df

Evaluate:   0%|          | 0/149 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

,system,level,precision_at_10,recall_at_10,mrr,ndcg_at_10,candidate_paragraph_count,chunk_precision_at_5,chunk_recall_at_5,chunk_mrr,chunk_ndcg_at_5
0,summary_hybrid_chunks5_para30_rerank10,paragraph,0.039597,0.079938,0.115229,0.065316,325.986577,NaN,NaN,NaN,NaN
1,summary_hybrid_chunks5,chunk_summary,NaN,NaN,NaN,NaN,NaN,0.190604,0.360768,0.415772,0.320503


In [15]:
summary_df

,system,level,precision_at_10,recall_at_10,mrr,ndcg_at_10,candidate_paragraph_count,chunk_precision_at_5,chunk_recall_at_5,chunk_mrr,chunk_ndcg_at_5
0,summary_hybrid_chunks5_para30_rerank10,paragraph,0.039597,0.079938,0.115229,0.065316,325.986577,NaN,NaN,NaN,NaN
1,summary_hybrid_chunks5,chunk_summary,NaN,NaN,NaN,NaN,NaN,0.190604,0.360768,0.415772,0.320503


## 13. Inspect one query

In [16]:
idx = 0
row = run_paragraph_rows[idx]
print("Query:", row["query"])
print("Novel:", row["novel_id"])
print("Gold docs:", row["gold_doc_ids"])
print("Selected chunks:", row["selected_chunk_ids"])
print("Candidate paragraphs:", row["candidate_paragraph_count"])
print("Top 10 after rerank:")
for rank, doc_id in enumerate(row["top10_after_rerank"], start=1):
    mark = "*" if doc_id in set(row["gold_doc_ids"]) else " "
    print(f"{rank:02d}{mark} {doc_id}: {corpus[doc_id]['text'][:300].replace(chr(10), ' ')}")

Query: Alyona advised Polo not to tell anyone when she left, which implies that:
Novel: 117
Gold docs: ['dqa-en-117-p101', 'dqa-en-117-p103', 'dqa-en-117-p107', 'dqa-en-117-p226', 'dqa-en-117-p229', 'dqa-en-117-p246', 'dqa-en-117-p441', 'dqa-en-117-p478', 'dqa-en-117-p92', 'dqa-en-117-p94', 'dqa-en-117-p98']
Selected chunks: ['dqa-en-117-c0021', 'dqa-en-117-c0014', 'dqa-en-117-c0019', 'dqa-en-117-c0003', 'dqa-en-117-c0013']
Candidate paragraphs: 362
Top 10 after rerank:
01  dqa-en-117-p883: Polo said, "Yes, because Mrs. Marshall asked me not to tell anyone I saw her this morning when she left the seaside. I immediately realized that there was trouble between her and her husband due to her relationship with Patrick Redfern. I thought she had a rendezvous with Patrick Redfern somewhere, 
02  dqa-en-117-p923: Linda said, "Yes." She added, "Alyona is very friendly to me."
03  dqa-en-117-p919: "Around one o'clock, I - later - I heard - about - Alyona..." Her voice wavered.
04  dqa-en-117-p1